# Auditing Content Moderation AI for Bias, Adversarial Robustness & Safety

End-to-end audit of a DistilBERT toxicity classifier on the Jigsaw Unintended Bias dataset.

## Section 1 — Setup and Data Loading

- Load 100k training rows + 20k evaluation rows (stratified on binarized `toxic`)
- Binarize `toxic` at threshold ≥ 0.5
- Inspect class balance and identity-column null summary

In [ ]:
!pip install -q transformers torch scikit-learn fairlearn aif360 pandas matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

USE_COLS = ["comment_text", "toxic", "black", "white", "muslim", "jewish"]
IDENTITY_COLS = ["black", "white", "muslim", "jewish"]

raw = pd.read_csv("data/jigsaw-unintended-bias-train.csv", usecols=USE_COLS)
raw["label"] = (raw["toxic"] >= 0.5).astype(int)

# Stratified 100k train / 20k eval split on the binarized label.
sample = raw.sample(n=120_000, random_state=SEED)
train_df, eval_df = train_test_split(
    sample, test_size=20_000, stratify=sample["label"], random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)

print("train:", train_df.shape, "eval:", eval_df.shape)
print("\nClass balance:")
print(pd.concat([
    train_df["label"].value_counts(normalize=True).rename("train"),
    eval_df["label"].value_counts(normalize=True).rename("eval"),
], axis=1).round(4))

print("\nIdentity null %:")
print(train_df[IDENTITY_COLS].isna().mean().round(4))